# Challenge 10 · The Hardware Architect
**Tier 3 - Advanced Challenges** · **Estimated time: 45-75 minutes** (an estimate, not a guarantee) · *Participant notebook*

> Your algorithm assumes every qubit can talk to every other. The chip's qubits sit on a line. Re-engineer the circuit for this hardware - cheaper, shallower, and still doing exactly the same thing.

### Learning objectives
- Measure circuit cost with depth and two-qubit gate count.
- Check that a circuit respects a coupling map and a basis-gate set.
- Use the transpiler (layout, routing, optimisation level, seeds) to reduce cost.
- Verify that an optimised circuit still implements the same unitary.

### Prerequisites
- Challenges 1-5 or equivalent
- Circuit depth and gate counts
- The idea of a coupling map (which qubits are connected)

### What you will write
| Function | Status |
|---|---|
| `circuit_cost` | required |
| `is_hardware_valid` | required |
| `optimize_for_hardware` | required |
| `cost_reduction` | required |
| `ghz_chain_rewrite` | bonus (optional) |

### Acceptance criteria
Your solution is accepted when all **core** and **edge-case** requirements hold:
- `circuit_cost` reports depth, two-qubit gate count and total gate count exactly as documented.
- `is_hardware_valid` accepts exactly the circuits that use only basis gates on coupled qubit pairs.
- `optimize_for_hardware` returns hardware-valid circuits.
- The optimised circuit implements the same unitary (up to global phase and the reported qubit layout).
- Optimisation is reproducible for a fixed seed.
- Invalid inputs (too many qubits, empty coupling map, negative seed) raise ValueError.
- `cost_reduction` returns (baseline - optimised) / baseline for the chosen metric: 0.0 for an empty baseline, negative for a worse circuit, ValueError for unknown metrics or bad cost dictionaries.

**Quality benchmarks** (reported separately from correctness - a stochastic optimiser is not required to be perfect):
- The optimised circuit is never worse than the baseline transpilation in two-qubit gates.
- On the supplied reference circuit the two-qubit gate count falls by at least 25% versus the baseline.

**Bonus (optional, never required to pass):**
- `ghz_chain_rewrite` hand-optimises the star-shaped GHZ preparation into a chain with n-1 two-qubit gates (state-equivalent).

**How this notebook works.** Run the cells from top to bottom. Cells marked **TODO** are yours. After each part run the **public checks**: they tell you what works and what does not. The instructors run additional hidden tests, so also try your own edge cases. Hints go from subtle to explicit - open them one at a time.

In [ ]:
# Setup - safe to re-run (works in Google Colab and in a local Jupyter session)
import importlib, importlib.util, pathlib, subprocess, sys

if importlib.util.find_spec('qiskit_fest') is None:
    here = pathlib.Path.cwd().resolve()
    # the project is the working folder, one of its parents, or (Colab: working folder /content) a folder right below it
    candidates = [here, *here.parents, *sorted(p for p in here.iterdir() if p.is_dir())]
    root = next((p for p in candidates
                 if (p / 'pyproject.toml').is_file() and (p / 'src' / 'qiskit_fest').is_dir()), None)
    if root is None:
        raise RuntimeError("Open this notebook from inside the qiskit-fall-fest-challenges folder "
                           "(see README.md, section 'Colab').")
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(root)])
    except subprocess.CalledProcessError:
        print('pip could not install the project (no internet?) - continuing with the libraries that are already installed.')
    # a running kernel does not re-read the .pth file that pip has just written, so make the package importable here and now
    sys.path.insert(0, str(root / 'src'))
    importlib.invalidate_caches()

from qiskit_fest.checks import run_public_checks, if_ready
from qiskit_fest.env import print_environment
print_environment()

## The problem
Your algorithm assumes that **every qubit can talk to every other qubit**. The chip it must run on is a **line**: only neighbouring qubits can run a two-qubit gate, and it only knows a handful of native gates. Moving quantum information around costs extra gates (SWAPs), and every two-qubit gate is the noisiest operation on the device. Re-engineer the circuit for this hardware - **cheaper, shallower, and still doing exactly the same thing**.

1. `circuit_cost` - measure a circuit: depth, two-qubit gates, total gates.
2. `is_hardware_valid` - does a circuit respect the basis gates and the coupling map?
3. `optimize_for_hardware` - transpile for the device and keep the best of several seeded attempts.
4. `cost_reduction` - how much cheaper than the baseline is your result?
5. *(Bonus)* `ghz_chain_rewrite` - beat the transpiler by hand on the GHZ circuit.

### The device model (used by the notebook and by every test)
* Qubits `0 … n−1`; the **coupling map** is a list of pairs `(a, b)` that can run a two-qubit gate. Couplings are **bidirectional**. The number of device qubits is `max(node) + 1`, and the circuits of this challenge have **exactly that many qubits** (otherwise `ValueError`).
* The **basis gates** are the only gates the chip runs (default `BASIS_GATES = ['rz', 'sx', 'x', 'cx']`). `measure` and `barrier` are always allowed.
* `LINE6 = [(0,1), (1,2), (2,3), (3,4), (4,5)]`. The helper `validate_device(coupling_edges, basis_gates)` is **provided**: it raises `ValueError` for an invalid device (empty coupling map, repeated or self couplings, empty basis) and returns `(edges, basis, num_qubits)`.
* The **baseline** is `baseline_transpile(circuit, edges, basis, seed)` (provided): Qiskit's lowest optimisation level - trivial layout, no optimisation - with the same device and seed.

### Function contracts
| Function | Input | Output | Raises `ValueError` when |
|---|---|---|---|
| `circuit_cost(circuit)` | a `QuantumCircuit` (counted **as it is**, nothing is decomposed) | `dict` with `depth` = `circuit.depth()` (Qiskit's definition: the longest chain of dependent operations, barriers do not count, measurements do), `two_qubit_gates` = gates acting on exactly two qubits, `total_gates` = all gates; **barriers, measurements and resets are not gates**. A composite gate counts as one gate | not a `QuantumCircuit` |
| `is_hardware_valid(circuit, coupling_edges, basis_gates)` | circuit, device | `bool`: `True` iff every instruction is a barrier, a measurement or a basis gate, every gate acts on qubits `< num_qubits` of the device, no gate acts on three or more qubits, and every two-qubit gate acts on a coupled pair (either direction) | invalid device, not a circuit |
| `optimize_for_hardware(circuit, coupling_edges, basis_gates=BASIS_GATES, seed=0, attempts=8)` | a unitary circuit with exactly as many qubits as the device | a `QuantumCircuit` that is hardware valid and implements the **same unitary** (up to a global phase) *including the layout and routing permutation the transpiler records in `circuit.layout`* - return the transpiler's circuit as it is. Equal arguments give equal circuits | too many / too few qubits, invalid device, `seed` not a non-negative integer, `attempts` not a positive integer |
| `cost_reduction(baseline, optimised, metric='two_qubit_gates')` | two cost dictionaries, `metric` in `depth`, `two_qubit_gates`, `total_gates` | `float` (baseline − optimised) / baseline; `0.0` if the baseline value is 0; negative if the result is *worse* | unknown metric, missing key, negative or non-finite value |
| *(bonus)* `ghz_chain_rewrite(n)` | integer n ≥ 2 | circuit on n qubits using only `h` and `cx`: H on qubit 0 and `n − 1` CNOTs between **neighbouring** qubits, preparing (\|0…0⟩ + \|1…1⟩)/√2 from \|0…0⟩ | n not an integer ≥ 2 |

**Correctness versus quality.** Correctness (exact checks): costs counted exactly as documented, validity decided exactly, the optimised circuit is valid, reproducible and equivalent (`Operator.from_circuit(...)` equals the original unitary up to a global phase, tolerance 1e-8). **Quality** (reported separately, never part of correctness): your result is *never worse than the baseline* in two-qubit gates on a set of test circuits and devices, and on the supplied reference circuit it needs **at least 25 % fewer two-qubit gates** than the baseline. The tests never look at the order or the names of gates inside your circuit.

## Concept: the cost of a limited chip
* **Connectivity.** A CNOT between qubits that are not neighbours must be preceded by **SWAPs** that bring them together; one SWAP is three CNOTs.
* **Layout** decides which virtual qubit lives on which physical qubit - a good layout can make SWAPs unnecessary. **Routing** inserts the SWAPs that remain. Both are heuristics that depend on random choices, so the transpiler takes a **seed**.
* **Optimisation levels** 0-3 add more clean-up (merging single-qubit gates, cancelling gate pairs, smarter layouts). **Best of several seeds** is a cheap way to tame the randomness.
* **Equivalence.** The transpiled circuit lives on *physical* qubits and may end with the qubits permuted by routing; `circuit.layout` records both, and `Operator.from_circuit` takes them into account.
* **Cost metrics.** Two-qubit gates dominate the error budget; depth bounds the runtime (and the decoherence). Total gate count is the third, weaker indicator.

In [ ]:
import numbers

from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_fest.challenges.c10_transpilation import BASIS_GATES, baseline_transpile, symmetric_coupling_map, validate_device

In [ ]:
from qiskit.quantum_info import Operator
from qiskit_fest.challenges.c10_transpilation import (
    LINE6, plot_coupling_map, plot_cost_comparison, reference_circuit, star_ghz)
from qiskit_fest.viz import draw_circuit

circuit = reference_circuit()
plot_coupling_map(LINE6, "The chip: a line of six qubits")
draw_circuit(circuit, "The reference circuit (written as if all qubits were connected)")
print("native gates of the chip:", BASIS_GATES)

## Part 1 · Measure a circuit  *(TODO)*
Complete `circuit_cost(circuit)`: `depth`, `two_qubit_gates` and `total_gates` of the circuit **as it is** (no decomposition). Barriers, measurements and resets are not gates.

<details><summary>💡 Hint 1 (subtle)</summary>

`circuit.data` lists the instructions; each has an `operation` (with a `name`) and `qubits`. Which names are not gates? How many qubits does a *two-qubit gate* have - and how many does a barrier on two qubits have?
</details>

<details><summary>💡 Hint 2</summary>

Keep only the instructions whose `operation.name` is not `barrier`, `measure`, `reset` or `delay`; count them for `total_gates` and count those with `len(inst.qubits) == 2` for `two_qubit_gates`. The depth is `circuit.depth()`.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

`gates = [inst for inst in circuit.data if inst.operation.name not in ("barrier", "measure", "reset", "delay")]`, then `len(gates)` and `sum(1 for g in gates if len(g.qubits) == 2)`.
</details>

In [ ]:
def circuit_cost(circuit: QuantumCircuit) -> dict:
    """{'depth', 'two_qubit_gates', 'total_gates'} of the circuit as given; barriers, measurements and resets are not gates."""
    # TODO: validate the argument (ValueError), count the instructions.
    raise NotImplementedError("TODO: implement circuit_cost()")

In [ ]:
with if_ready():
    baseline = baseline_transpile(circuit, LINE6, BASIS_GATES, seed=0)
    print("as written     :", circuit_cost(circuit))
    print("baseline (lvl 0):", circuit_cost(baseline))

In [ ]:
report = run_public_checks(10, globals(), keyword="circuit_cost")

## Part 2 · Does it fit the chip?  *(TODO)*
Complete `is_hardware_valid(circuit, coupling_edges, basis_gates)`. A circuit is valid iff every instruction is a barrier, a measurement or a basis gate, no gate acts on three or more qubits, every gate acts on device qubits and every two-qubit gate acts on a **coupled** pair (either direction). Use the provided `validate_device` first.

<details><summary>💡 Hint 1 (subtle)</summary>

Three questions per instruction: *is the gate allowed*, *how many qubits does it touch*, *are those qubits neighbours*? `circuit.find_bit(qubit).index` turns a `Qubit` into its integer index.
</details>

<details><summary>💡 Hint 2</summary>

Store the couplings as a set of `frozenset({a, b})` so that direction does not matter. `measure` and `barrier` are always fine; a barrier may touch any qubits.
</details>

In [ ]:
def is_hardware_valid(circuit: QuantumCircuit, coupling_edges, basis_gates) -> bool:
    """True iff the circuit uses only barriers, measurements and basis gates, on device qubits, with every two-qubit gate on a coupled pair."""
    # TODO: validate_device, then check every instruction.
    raise NotImplementedError("TODO: implement is_hardware_valid()")

In [ ]:
with if_ready():
    print("reference circuit as written:", is_hardware_valid(circuit, LINE6, BASIS_GATES), "(it uses h and non-neighbouring CNOTs)")
    print("baseline transpilation      :", is_hardware_valid(baseline_transpile(circuit, LINE6, BASIS_GATES, 0), LINE6, BASIS_GATES))

In [ ]:
report = run_public_checks(10, globals(), keyword="is_hardware_valid")

## Part 3 · Optimise for the chip  *(TODO)*
Complete `optimize_for_hardware(circuit, coupling_edges, basis_gates, seed=0, attempts=8)`: transpile the circuit for the device and return the **best** result, judged by (two-qubit gates, depth, total gates). It must be valid, equivalent and reproducible; it should never be worse than the baseline.

<details><summary>💡 Hint 1 (subtle)</summary>

The transpiler is randomised and its optimisation levels differ. If you can generate several candidates cheaply, which one do you keep - and can you make sure that you are never worse than the baseline?
</details>

<details><summary>💡 Hint 2</summary>

`generate_preset_pass_manager(optimization_level=3, coupling_map=symmetric_coupling_map(edges), basis_gates=basis, seed_transpiler=s).run(circuit)` gives one candidate for seed `s`. Use the seeds `seed, seed + 1, …, seed + attempts − 1` (deterministic!) and put the baseline in the candidate list too. Validate the device with `validate_device` and compare the circuit's width with the device size.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

`candidates = [baseline_transpile(circuit, edges, basis, seed)] + [pass_manager(seed + k).run(circuit) for k in range(attempts)]`, then `min(candidates, key=lambda c: (cost["two_qubit_gates"], cost["depth"], cost["total_gates"]))` with `cost = circuit_cost(c)`. Do not strip the circuit's `layout`.
</details>

In [ ]:
def optimize_for_hardware(circuit: QuantumCircuit, coupling_edges, basis_gates=BASIS_GATES, seed: int = 0, attempts: int = 8) -> QuantumCircuit:
    """Best of several seeded transpilations (and the baseline): valid, equivalent, reproducible, never worse than the baseline."""
    # TODO: validate (ValueError), generate candidates, keep the cheapest by (two-qubit gates, depth, total gates).
    raise NotImplementedError("TODO: implement optimize_for_hardware()")

In [ ]:
with if_ready():
    optimised = optimize_for_hardware(circuit, LINE6, BASIS_GATES, seed=0)
    rows = {"baseline": circuit_cost(baseline_transpile(circuit, LINE6, BASIS_GATES, 0)), "optimised": circuit_cost(optimised)}
    print("valid for the chip:", is_hardware_valid(optimised, LINE6, BASIS_GATES), "| baseline:", rows["baseline"], "| optimised:", rows["optimised"])
    print("same unitary as the original (up to a global phase and the layout):", Operator.from_circuit(optimised).equiv(Operator(circuit), atol=1e-8))
    plot_cost_comparison(rows, "Reference circuit on a line of six qubits")

In [ ]:
report = run_public_checks(10, globals(), keyword="optimize_for_hardware")

## Part 4 · How much did you save?  *(TODO)*
Complete `cost_reduction(baseline, optimised, metric='two_qubit_gates')`: the **fraction** by which the optimised circuit is cheaper: (baseline − optimised) / baseline. `0.0` if the baseline value is 0; negative if you made things worse.

<details><summary>💡 Hint</summary>

Check that `metric` is one of the three keys and that both dictionaries contain it with a non-negative finite number; then it is one line of arithmetic.
</details>

In [ ]:
def cost_reduction(baseline: dict, optimised: dict, metric: str = "two_qubit_gates") -> float:
    """(baseline - optimised) / baseline for the chosen metric; 0.0 when the baseline value is 0."""
    # TODO: validate (ValueError), return the relative saving.
    raise NotImplementedError("TODO: implement cost_reduction()")

In [ ]:
with if_ready():
    base_cost = circuit_cost(baseline_transpile(circuit, LINE6, BASIS_GATES, 0))
    best_cost = circuit_cost(optimize_for_hardware(circuit, LINE6, BASIS_GATES, seed=0))
    for metric in ("two_qubit_gates", "depth", "total_gates"):
        print(f"{metric:16s} reduction versus the baseline: {cost_reduction(base_cost, best_cost, metric):6.1%}")

In [ ]:
report = run_public_checks(10, globals(), keyword="cost_reduction")

## Part 5 · Experiments  *(not graded)*
How far does the transpiler get on the GHZ circuit written as a star (one CNOT from qubit 0 to every other qubit)? A line needs SWAPs for it. Compare with the lower bound `n − 1` CNOTs.

In [ ]:
with if_ready():
    print(f"{'n':>2} {'baseline':>9} {'optimised':>10} {'chain (n-1)':>12}")
    for n in (4, 6, 8):
        line = [(i, i + 1) for i in range(n - 1)]
        star = star_ghz(n)
        b = circuit_cost(baseline_transpile(star, line, BASIS_GATES, 0))["two_qubit_gates"]
        o = circuit_cost(optimize_for_hardware(star, line, BASIS_GATES, 0))["two_qubit_gates"]
        print(f"{n:>2} {b:>9} {o:>10} {n - 1:>12}")

## Bonus · Beat the transpiler by hand  *(optional)*
The transpiler cannot know that the GHZ state can be prepared along a **chain**. Complete `ghz_chain_rewrite(n)`: H on qubit 0 and `n − 1` CNOTs between neighbouring qubits, preparing (|0…0⟩ + |1…1⟩)/√2.

<details><summary>💡 Hint</summary>

Instead of copying qubit 0 onto every other qubit, copy each qubit onto the *next* one: CNOT(0, 1), then CNOT(1, 2), and so on.
</details>

In [ ]:
def ghz_chain_rewrite(n: int) -> QuantumCircuit:
    """GHZ preparation with H on qubit 0 and n - 1 CNOTs between neighbouring qubits (only h and cx)."""
    # TODO (bonus): validate n (ValueError), build the chain.
    raise NotImplementedError("TODO (bonus): implement ghz_chain_rewrite()")

In [ ]:
with if_ready():
    for n in (4, 6, 8):
        line = [(i, i + 1) for i in range(n - 1)]
        chain = ghz_chain_rewrite(n)
        print(f"n={n}: chain uses {circuit_cost(chain)['two_qubit_gates']} CNOTs, valid on the line apart from the h gate: "
              f"{is_hardware_valid(chain, line, BASIS_GATES + ['h'])}")

In [ ]:
report = run_public_checks(10, globals(), keyword="bonus")

## Reflection
1. The baseline needed many more CNOTs than the optimised circuit. Which two transpiler decisions (layout, routing, clean-up) contributed most for the reference circuit? How would you find out?
2. Why does the transpiler take a *seed*, and why is "best of 8 seeds" a legitimate (and cheap) trick? What would be *wrong* about choosing the best seed by looking at the **output of the algorithm** instead of the gate count?
3. Why is the unitary of the transpiled circuit only equal to the original *up to a global phase and the layout*?
4. The chain GHZ circuit beats the transpiler. What does this say about the limits of automatic optimisation, and where else could a hand rewrite help?
5. A SWAP costs three CNOTs. If the chip's two-qubit error rate is 1 %, what is the probability that the SWAPs *alone* destroy a circuit that needs ten of them?

### Hand-in
When you are done, save your functions as a file (or simply hand in this notebook).

In [ ]:
# Optional: save your functions as a stand-alone file you can hand in
from qiskit_fest.export import export_submission
export_submission(globals(), 10)